In [1]:
import faiss
from datasets import load_dataset
import pandas as pd
import numpy as np

c:\Users\nithishkumar\Personal Data\Projects\overall\prototyping\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
movies_dataset_name = 'acloudfan/embedded_movies_small'

movies_dataset = load_dataset(movies_dataset_name)

# This will hold the data for movies, will be cross referenced for details
movies_dataset_train = movies_dataset['train']
# Embeddings need to be in numpy array with dtype=float32
movies_dataset_train_np = np.array(movies_dataset_train['plot_embedding']).astype(np.float32)

# This will hold the details for test dataset
movies_dataset_test = movies_dataset['test']
movies_dataset_test_np = np.array(movies_dataset_test['plot_embedding']).astype(np.float32)

In [2]:
# Check the embedding dimension
embeddings_dimension = len(movies_dataset_test_np[0])

embeddings_dimension

NameError: name 'movies_dataset_test_np' is not defined

In [ ]:
# Utility method to print the movie information
def  print_movie(movie):
    print('title = ', movie['title'])
    print('genres = ',movie['genres'])
    print('fullplot = ', movie['fullplot'])

# Utility method to run search and print results
# Returns the indexes
def   query_embeddings(faiss_index, k, test_index):
    query_embedding = movies_dataset_test_np[test_index]
    query_embedding = np.expand_dims(query_embedding, axis=0)

    result_indexes = []
    
    print('Query Result')
    print('-----------------')
    print_movie(movies_dataset_test[test_index])

    distances, movie_indexes = faiss_index.search(query_embedding, k)

    print(distances)
    for i, movie_index in enumerate(movie_indexes[0]):
        result_indexes.append(movie_index)
        print(i,'--',movie_index,'-- Distance = ', distances[0][i],'---')
        print_movie(movies_dataset_train[movie_index.item()])

    return result_indexes
        

In [ ]:
# Create the index
flatl2_index = faiss.IndexFlatL2(embeddings_dimension)

# Add the training embeddings to the index
flatl2_index.add(movies_dataset_train_np)

# Check if index needs training
flatl2_index.is_trained

In [ ]:
%%time

# Change the index to try out different movies ~400 rows
test_movie_index = 14

# Change the value of k as needed
k = 2

# Test for a few movies
baseline_result_indexes = query_embeddings(flatl2_index,k,test_movie_index)

print('-----Query Movie----')
print(movies_dataset_train[test_movie_index]['fullplot'])
print('--------------------')
print("Baseline indexes = ", baseline_result_indexes)